# Grievance Intelligence Pipeline (v5) — deployable class + full evaluation + error analysis

Open-set, bilingual (Hindi / English / Hinglish) grievance category classifier, packaged as a
`GrievanceModel` class ready for integration into a backend system.

**Target categories:** Electricity, Water, Road, Sanitation, Health, Other

**Libraries used:** pandas, numpy, sklearn, scipy, re — no deep learning, no external APIs.

---
**Data note (carried over from v4, still true here):** `Citizen.csv` (the real training file) contains **six** categories — Roads, Water, Electricity, Sanitation, Healthcare, and **Transport**. The requested target list only names five known categories + Other (no Transport).

Dropping the Transport rows would throw away ~17% of real training data and would force every real transport complaint into `'Other'`. Instead:
- the model trains on all 6 real classes,
- `'Roads' → 'Road'` and `'Healthcare' → 'Health'` are renamed for output,
- `'Transport'` is kept as its own visible class rather than hidden.

Change `CATEGORY_RENAME` below if you'd rather fold Transport into `'Other'`.

In [1]:
import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

## Config

In [2]:
CATEGORY_RENAME = {
    'Roads': 'Road',
    'Healthcare': 'Health',
    'Water': 'Water',
    'Electricity': 'Electricity',
    'Sanitation': 'Sanitation',
    'Transport': 'Transport',   # see DATA NOTE above
}

DEFAULT_CONFIDENCE_THRESHOLD = 0.50
DEFAULT_MARGIN_THRESHOLD = 0.25

## Text normalization + domain keyword dictionaries

Hindi/Marathi (Devanagari) and Romanized Hinglish spelling variants are collapsed onto a single
canonical **English** token per concept, before feature extraction and before domain-keyword
matching. This is a fixed lookup table, not real translation — anything not listed passes through
unchanged (a limitation discussed in the Error Analysis section below).

In [3]:
NORMALIZATION_MAP = {
    'bijli': 'electricity', 'बिजली': 'electricity', 'वीज': 'electricity',
    'vij': 'electricity', 'current': 'electricity', 'transformer': 'electricity',
    'power': 'electricity', 'voltage': 'electricity',

    'pani': 'water', 'paani': 'water', 'पानी': 'water', 'पाणी': 'water',
    'jal': 'water', 'jal board': 'water',

    'sadak': 'road', 'सड़क': 'road', 'रस्त्यावर': 'road', 'रस्ता': 'road',
    'gaddha': 'pothole', 'gaddhe': 'pothole', 'gaddho': 'pothole',
    'speed breaker': 'speedbreaker', 'speedbreaker': 'speedbreaker',

    'kachra': 'garbage', 'कचरा': 'garbage', 'safai': 'sanitation', 'सफाई': 'sanitation',
    'ganda': 'dirty', 'gandagi': 'dirty', 'kooda': 'garbage', 'कूड़ा': 'garbage',

    'aspatal': 'hospital', 'अस्पताल': 'hospital', 'रुग्णालय': 'hospital',
    'dawai': 'medicine', 'mareez': 'patient', 'chikitsa': 'treatment',

    'bus': 'bus', 'बस': 'bus', 'rickshaw': 'rickshaw',
}
_NORM_KEYS_SORTED = sorted(NORMALIZATION_MAP.keys(), key=len, reverse=True)
_NORM_PATTERN = re.compile(r'\b(' + '|'.join(re.escape(k) for k in _NORM_KEYS_SORTED) + r')\b')

DOMAIN_KEYWORDS = {
    'Electricity': ['electricity', 'meter reading', 'load shedding', 'fuse', 'wire'],
    'Water': ['water', 'pipeline', 'pipe', 'tap', 'tanker', 'leakage', 'leak',
              'contaminated', 'drinking water', 'borewell'],
    'Road': ['road', 'pothole', 'street', 'highway', 'flyover', 'bridge', 'traffic',
             'speedbreaker', 'signage', 'overbridge', 'footpath', 'divider'],
    'Sanitation': ['garbage', 'sanitation', 'dirty', 'sewage', 'drain', 'dustbin',
                   'waste', 'cleanliness', 'sweeper'],
    'Health': ['hospital', 'doctor', 'health', 'medicine', 'treatment', 'ambulance',
               'clinic', 'nurse', 'patient', 'blood bank', 'blood group'],
    'Transport': ['bus', 'transport', 'route', 'auto', 'rickshaw', 'train',
                  'station', 'conductor', 'bus stop', 'taxi'],
}
_DOMAIN_PATTERNS = {
    domain: [re.compile(r'\b' + re.escape(w) + r'\b') for w in words]
    for domain, words in DOMAIN_KEYWORDS.items()
}

In [4]:
def normalize_hinglish(text):
    """Replaces every known Hindi/Hinglish/Marathi variant with its canonical
    English token, using word-boundary regex matching (never a substring hack)."""
    return _NORM_PATTERN.sub(lambda m: NORMALIZATION_MAP[m.group(0)], text)

def clean_text(text):
    """
    1. Lowercase.
    2. Apply Hindi/Hinglish -> English normalization dictionary.
    3. Strip everything except a-z, 0-9, Devanagari block, whitespace (so
       un-normalized Devanagari still survives for the char n-gram features).
    4. Collapse repeated whitespace.
    """
    text = str(text).lower()
    text = normalize_hinglish(text)
    text = re.sub(r'[^a-z0-9\u0900-\u097F\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def detect_domain(clean_txt):
    """
    Counts keyword hits per domain on already-cleaned/normalized text.
    Returns (best_domain_or_None, hit_count, all_domain_scores).

    A domain only "wins" if it has strictly more hits than every other domain
    (an unambiguous strong match). A tie is treated as ambiguous ON PURPOSE --
    we'd rather let the ML classifier arbitrate than have the keyword filter
    silently guess between two equally-matched domains.
    """
    scores = {domain: sum(1 for p in patterns if p.search(clean_txt))
              for domain, patterns in _DOMAIN_PATTERNS.items()}
    max_hits = max(scores.values())
    if max_hits == 0:
        return None, 0, scores
    top_domains = [d for d, s in scores.items() if s == max_hits]
    if len(top_domains) > 1:
        return None, max_hits, scores
    return top_domains[0], max_hits, scores

## Step 1: Data Understanding

Before any modeling decision, we need to know: how many samples, how balanced are the classes, and
crucially — how many are exact duplicates of each other, since that determines whether a normal
random split is even safe to use.

In [5]:
def explore_data(train_df):
    """Prints the basic health-check every NLP project needs before modeling:
    sample count, class balance, and duplicate rate -- because all three
    directly determine which modeling choices (below) are actually safe."""
    print(f"Total samples: {len(train_df)}")
    print(f"\nClass distribution:\n{train_df['category'].value_counts()}")

    n_unique = train_df['text'].nunique()
    n_total = len(train_df)
    dup_rate = 1 - (n_unique / n_total)
    print(f"\nUnique complaint texts: {n_unique} / {n_total} total rows "
          f"({dup_rate:.1%} are duplicates of another row)")
    print(
        "\nWHY DUPLICATES ARE DANGEROUS (data leakage):\n"
        "If the exact same sentence appears in both the train and test split, the\n"
        "model doesn't need to generalize -- it can just memorize that one sentence\n"
        "and 'predict' it perfectly at test time. That inflates accuracy to a number\n"
        "that has nothing to do with how the model will perform on a genuinely new\n"
        "complaint in production. With only a few dozen unique templates repeated\n"
        "many times each (as is the case here), a naive random row-level split will\n"
        "almost certainly put duplicates on both sides -- so a GROUP-based split,\n"
        "keyed on the unique text, is mandatory here, not optional."
    )

raw_df = pd.read_csv('Citizen.csv', skiprows=1)
raw_df.columns = ['text', 'category']
raw_df['category'] = raw_df['category'].map(CATEGORY_RENAME).fillna(raw_df['category'])
explore_data(raw_df)

Total samples: 1200

Class distribution:
category
Road           211
Transport      206
Sanitation     200
Water          197
Health         194
Electricity    192
Name: count, dtype: int64

Unique complaint texts: 30 / 1200 total rows (97.5% are duplicates of another row)

WHY DUPLICATES ARE DANGEROUS (data leakage):
If the exact same sentence appears in both the train and test split, the
model doesn't need to generalize -- it can just memorize that one sentence
and 'predict' it perfectly at test time. That inflates accuracy to a number
that has nothing to do with how the model will perform on a genuinely new
complaint in production. With only a few dozen unique templates repeated
many times each (as is the case here), a naive random row-level split will
almost certainly put duplicates on both sides -- so a GROUP-based split,
keyed on the unique text, is mandatory here, not optional.


## Step 3: Leak-Free Train/Test Split

**Why this matters:** `Citizen.csv` has only ~30 unique complaint sentences, each repeated ~40x. A
normal row-level split puts the same sentence in both train and test, so the model just memorizes
it — producing a fake ~100% accuracy that says nothing about real generalization.

**Fix:** split at the unique-text level (group-based split), stratified by category, then expand
back to all duplicate rows, keeping every duplicate of a sentence on the **same** side of the
split. This is equivalent in effect to sklearn's `GroupShuffleSplit`, grouped by `clean_text`.

In [6]:
def leak_free_split(df, text_col='clean_text', label_col='category', test_size=0.2, seed=42):
    unique_df = df.drop_duplicates(subset=text_col)[[text_col, label_col]]
    train_unique, test_unique = train_test_split(
        unique_df, test_size=test_size, random_state=seed, stratify=unique_df[label_col]
    )
    train_text_set = set(train_unique[text_col])
    test_text_set = set(test_unique[text_col])
    train_rows = df[df[text_col].isin(train_text_set)]
    test_rows = df[df[text_col].isin(test_text_set)]
    return train_rows, test_rows

## Step 4: Feature Engineering

- **word-level TF-IDF (1–2 grams)** → captures **meaning** / topic vocabulary (`'water'`, `'power cut'`, `'pothole'`). This is what tells the model *what* the complaint is about.
- **char-level TF-IDF (3–5 grams, `char_wb`)** → captures **spelling variation**. Hinglish has no fixed spelling (`'paani'` vs `'pani'` vs `'panee'`); a word-level model treats these as three unrelated tokens, but they share overlapping 3–5 character chunks (`'pan'`, `'aani'`), so the char-level model can still recognize the connection even for a spelling it has never seen exactly.

Both are fit on **train text only** — fitting on test/holdout text would leak that vocabulary into the model's feature space before evaluation.

In [7]:
def build_vectorizers(train_texts):
    word_vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1)
    char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=1, max_features=1500)
    word_vec.fit(train_texts)
    char_vec.fit(train_texts)
    return word_vec, char_vec

def vectorize(texts, word_vec, char_vec):
    """Combines word + char features into one matrix via scipy hstack --
    both feature sets are used together by the final model."""
    return hstack([word_vec.transform(texts), char_vec.transform(texts)])

## Step 11: Deployable Class

Everything from Steps 2–9 (cleaning, leak-free split, TF-IDF features, Logistic Regression, domain
override, confidence/margin gate) is wrapped into one class so it can be dropped into a backend
service:

```python
model = GrievanceModel()
model.train('Citizen.csv')
model.predict("bijli 2 din se nahi hai")
# -> {'category': 'Electricity', 'confidence': 0.97, 'reason': 'domain override match (1 keyword hit(s))'}
```

In [8]:
class GrievanceModel:
    """
    End-to-end grievance category classifier, packaged for integration into a
    larger backend system.
    """

    def __init__(self, confidence_threshold=DEFAULT_CONFIDENCE_THRESHOLD,
                 margin_threshold=DEFAULT_MARGIN_THRESHOLD):
        self.confidence_threshold = confidence_threshold
        self.margin_threshold = margin_threshold
        self.word_vec = None
        self.char_vec = None
        self.clf = None
        self.train_rows_ = None
        self.test_rows_ = None
        self.is_trained = False

    # ---- training ----
    def train(self, train_path='Citizen.csv', test_size=0.2, seed=42, verbose=True):
        """Loads Citizen.csv, cleans it, performs a leak-free split, builds
        TF-IDF features, and fits the Logistic Regression classifier."""
        train_df = pd.read_csv(train_path, skiprows=1)
        train_df.columns = ['text', 'category']
        train_df['category'] = train_df['category'].map(CATEGORY_RENAME).fillna(train_df['category'])
        train_df['clean_text'] = train_df['text'].apply(clean_text)

        train_rows, test_rows = leak_free_split(train_df, test_size=test_size, seed=seed)
        self.word_vec, self.char_vec = build_vectorizers(train_rows['clean_text'])
        X_train = vectorize(train_rows['clean_text'], self.word_vec, self.char_vec)

        # class_weight='balanced' compensates for the mild class imbalance in
        # the real data (e.g. Roads: 211 rows vs Electricity: 192 rows).
        self.clf = LogisticRegression(max_iter=2000, class_weight='balanced')
        self.clf.fit(X_train, train_rows['category'])

        self.train_rows_ = train_rows
        self.test_rows_ = test_rows
        self.is_trained = True

        if verbose:
            print(f"Trained on {X_train.shape[0]} rows ({train_rows['clean_text'].nunique()} "
                  f"unique templates), {len(self.clf.classes_)} categories: {list(self.clf.classes_)}")
            print(f"Held out {len(test_rows)} rows ({test_rows['clean_text'].nunique()} "
                  f"unique templates) for leak-free evaluation.")
        return self

    # ---- single-text prediction (Step 9 final prediction logic) ----
    def predict(self, text):
        """
        Decision order:
          1. Clean + normalize text.
          2. Rule-based domain detection. An UNAMBIGUOUS strong keyword match
             OVERRIDES the ML model -- this is what makes genuinely unseen
             phrasings of a KNOWN topic (e.g. "transformer blast") still route
             correctly even when the tiny classifier is unsure or wrong.
          3. Otherwise, run the classifier and gate its answer: top-1
             probability >= confidence_threshold AND margin over the runner-up
             >= margin_threshold. Failing either -> 'Other' (a guess we don't
             trust is worse than admitting uncertainty).
        Returns: {'category': str, 'confidence': float, 'reason': str}
        """
        if not self.is_trained:
            raise RuntimeError("Call .train() before .predict().")

        clean = clean_text(text)
        domain, hits, _ = detect_domain(clean)
        vec = vectorize([clean], self.word_vec, self.char_vec)
        proba = self.clf.predict_proba(vec)[0]

        if domain is not None:
            if domain in self.clf.classes_:
                model_conf = float(proba[list(self.clf.classes_).index(domain)])
            else:
                model_conf = None
            keyword_conf = min(0.95, 0.6 + 0.1 * (hits - 1))
            confidence = max(keyword_conf, model_conf) if model_conf is not None else keyword_conf
            return {'category': domain, 'confidence': round(confidence, 3),
                    'reason': f'domain override match ({hits} keyword hit(s))'}

        order = np.argsort(proba)[::-1]
        top1_prob, top2_prob = proba[order[0]], proba[order[1]]
        top1_class = self.clf.classes_[order[0]]
        margin = top1_prob - top2_prob

        if top1_prob < self.confidence_threshold:
            return {'category': 'Other', 'confidence': round(float(top1_prob), 3),
                    'reason': f'low confidence ({top1_prob:.2f} < {self.confidence_threshold})'}
        if margin < self.margin_threshold:
            return {'category': 'Other', 'confidence': round(float(top1_prob), 3),
                    'reason': f'top-2 classes too close (margin {margin:.2f} < {self.margin_threshold})'}
        return {'category': top1_class, 'confidence': round(float(top1_prob), 3),
                'reason': 'confident model prediction'}

    # ---- batch prediction helper ----
    def predict_batch(self, texts):
        return [self.predict(t) for t in texts]

    # ---- raw classifier accuracy, no gate (diagnostic only) ----
    def _raw_predict(self, clean_texts):
        vec = vectorize(clean_texts, self.word_vec, self.char_vec)
        return self.clf.predict(vec)

## Training

In [9]:
model = GrievanceModel()
model.train('Citizen.csv')

Trained on 958 rows (24 unique templates), 6 categories: ['Electricity', 'Health', 'Road', 'Sanitation', 'Transport', 'Water']
Held out 242 rows (6 unique templates) for leak-free evaluation.


## Step 6 + 10: Evaluation

Full evaluation suite:
1. **Train vs test accuracy** (raw classifier, no gate) — the gap between them is the honest signal of overfitting on this tiny dataset.
2. **Classification report** (precision / recall / f1) on the leak-free test set — accuracy alone hides which categories the model is actually weak on.
3. **Confusion matrix** — shows which categories get confused with which.
4. **Gated pipeline results** (Correct / Other / Wrong) on the leak-free test set.

In [10]:
def evaluate_model(model):
    train_rows, test_rows = model.train_rows_, model.test_rows_
    X_train = vectorize(train_rows['clean_text'], model.word_vec, model.char_vec)
    X_test = vectorize(test_rows['clean_text'], model.word_vec, model.char_vec)

    train_pred = model.clf.predict(X_train)
    test_pred = model.clf.predict(X_test)
    train_acc = accuracy_score(train_rows['category'], train_pred)
    test_acc = accuracy_score(test_rows['category'], test_pred)
    print("1. TRAIN vs TEST ACCURACY (raw classifier, no domain override / no gate)")
    print(f"Train accuracy: {train_acc:.2%}   |   Test accuracy: {test_acc:.2%}")
    print(
        f"Gap: {train_acc - test_acc:.2%}. A large gap would mean the model is "
        f"memorizing training phrasing rather than generalizing; a small gap on "
        f"a dataset this small is still not proof of real-world robustness -- "
        f"see the holdout-set numbers further down for that."
    )
    return {
        'train_acc': train_acc, 'test_acc': test_acc,
        'test_preds': test_pred, 'test_true': test_rows['category'].values,
    }

eval_results = evaluate_model(model)

1. TRAIN vs TEST ACCURACY (raw classifier, no domain override / no gate)
Train accuracy: 100.00%   |   Test accuracy: 81.40%
Gap: 18.60%. A large gap would mean the model is memorizing training phrasing rather than generalizing; a small gap on a dataset this small is still not proof of real-world robustness -- see the holdout-set numbers further down for that.


In [11]:
print("2. CLASSIFICATION REPORT (leak-free test set, raw classifier)")
print(classification_report(model.test_rows_['category'], eval_results['test_preds'], zero_division=0))

2. CLASSIFICATION REPORT (leak-free test set, raw classifier)
              precision    recall  f1-score   support

 Electricity       1.00      1.00      1.00        34
      Health       1.00      1.00      1.00        33
        Road       1.00      1.00      1.00        41
  Sanitation       0.49      1.00      0.66        43
   Transport       1.00      1.00      1.00        46
       Water       0.00      0.00      0.00        45

    accuracy                           0.81       242
   macro avg       0.75      0.83      0.78       242
weighted avg       0.72      0.81      0.75       242



In [12]:
# 3. Confusion matrix -- this is where the model's real confusions show up
# BEFORE the domain filter or gate hides them from the final output.
labels = sorted(model.test_rows_['category'].unique())
cm = confusion_matrix(model.test_rows_['category'], eval_results['test_preds'], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true:{l}" for l in labels], columns=[f"pred:{l}" for l in labels])
cm_df

,pred:Electricity,pred:Health,pred:Road,pred:Sanitation,pred:Transport,pred:Water
true:Electricity,34,0,0,0,0,0
true:Health,0,33,0,0,0,0
true:Road,0,0,41,0,0,0
true:Sanitation,0,0,0,43,0,0
true:Transport,0,0,0,0,46,0
true:Water,0,0,0,45,0,0


In [13]:
# 4. Gated pipeline results on the leak-free test set
gated_results = model.test_rows_['text'].apply(model.predict)
gated_preds = gated_results.apply(lambda r: r['category'])
true_labels = model.test_rows_['category']
correct = (gated_preds == true_labels)
other = (gated_preds == 'Other')
wrong = (~correct) & (~other)
print(f"4. GATED PIPELINE RESULTS -- leak-free test set (n = {len(model.test_rows_)})")
print(f"  Correct:         {correct.mean():.2%}")
print(f"  Sent to 'Other': {other.mean():.2%}")
print(f"  Wrong:           {wrong.mean():.2%}")

4. GATED PIPELINE RESULTS -- leak-free test set (n = 242)
  Correct:         100.00%
  Sent to 'Other': 0.00%
  Wrong:           0.00%


## Realistic manual test set

A small hand-written test set of realistic Hinglish complaints, none of which match the training templates word-for-word.

In [14]:
samples = [
    "bijli 2 din se nahi hai",
    "pani supply band hai",
    "transformer blast ho gaya",
    "road mein potholes hai",
    "hospital mein doctor nahi hai",
]
for s in samples:
    r = model.predict(s)
    print(f"[{r['category']:<12} | conf={r['confidence']:.2f}] {r['reason']:<45} -- {s}")

[Electricity  | conf=0.81] domain override match (1 keyword hit(s))      -- bijli 2 din se nahi hai
[Water        | conf=0.79] domain override match (1 keyword hit(s))      -- pani supply band hai
[Electricity  | conf=0.98] domain override match (1 keyword hit(s))      -- transformer blast ho gaya
[Road         | conf=0.85] domain override match (1 keyword hit(s))      -- road mein potholes hai
[Health       | conf=0.92] domain override match (2 keyword hit(s))      -- hospital mein doctor nahi hai


## Step 7: Error Analysis

Looks at every wrong prediction on the leak-free test set (raw classifier, no gate) and classifies
the likely cause into one of three buckets that matter for a low-data multilingual project like
this one:

- **unseen vocabulary** — the test sentence uses words that never appear in *any* training sentence at all (word-level TF-IDF has literally zero signal for them; the char-level features only help with near-misspellings, not entirely new words),
- **Hinglish spelling variation** — the sentence contains a normalization-map miss (a Hindi/Hinglish word not covered by `NORMALIZATION_MAP`), so it wasn't collapsed onto a canonical token and instead diluted the feature space,
- **small dataset** — neither of the above applies; the error is best explained by there simply being too few training examples per class for the linear model to have learned a clean decision boundary yet.

In [15]:
def error_analysis(model, eval_results):
    test_rows = model.test_rows_
    train_vocab = set(' '.join(model.train_rows_['clean_text']).split())

    test_preds = eval_results['test_preds']
    test_true = eval_results['test_true']
    wrong_mask = test_preds != test_true

    n_wrong = wrong_mask.sum()
    print(f"{n_wrong} / {len(test_rows)} test rows misclassified by the raw model.\n")

    if n_wrong == 0:
        print("No raw-classifier errors on this split. This is expected given how few "
              "unique templates exist -- it is NOT evidence the model generalizes well "
              "to genuinely novel phrasing; see the holdout-set and manual real-world "
              "test results for a harsher, more honest signal.")
        return

    wrong_rows = test_rows[wrong_mask].copy()
    wrong_rows['predicted'] = test_preds[wrong_mask]
    wrong_rows['true'] = test_true[wrong_mask]
    # Rows are duplicated ~40x per unique template (see Step 1) -- collapse to
    # one line per unique sentence so the analysis is readable, but keep the
    # occurrence count so the scale of the error is still visible.
    wrong_unique = (
        wrong_rows.groupby(['text', 'clean_text', 'predicted', 'true'])
        .size().reset_index(name='count')
    )

    for _, row in wrong_unique.iterrows():
        tokens = row['clean_text'].split()
        unseen_tokens = [t for t in tokens if t not in train_vocab]
        has_unnormalized_hindi = bool(re.search(r'[\u0900-\u097F]', row['clean_text']))

        if unseen_tokens and len(unseen_tokens) >= max(1, len(tokens) // 2):
            cause = f"unseen vocabulary (never-seen tokens: {unseen_tokens})"
        elif has_unnormalized_hindi:
            cause = "Hinglish/Hindi word not covered by the normalization dictionary"
        else:
            cause = "small dataset -- too few examples of this pattern for a clean decision boundary"

        print(f"  true={row['true']:<12} pred={row['predicted']:<12} (x{row['count']:>2} rows) | \"{row['text']}\"")
        print(f"    likely cause: {cause}\n")

    print(
        "TAKEAWAY: with only 24 unique training templates split across 6 classes, "
        "the model has seen roughly 4 sentence patterns per category. Any test "
        "sentence that departs from those patterns -- in vocabulary, in Hindi/Hinglish "
        "spelling, or just in sentence structure -- is operating outside what the "
        "linear model can reliably generalize from. This is exactly why the "
        "domain-keyword layer and the confidence/margin gate exist: they are "
        "not optional polish, they are load-bearing given how little training data "
        "there actually is."
    )

error_analysis(model, eval_results)

45 / 242 test rows misclassified by the raw model.

  true=Water        pred=Sanitation   (x45 rows) | "Pipeline leakage near my house"
    likely cause: unseen vocabulary (never-seen tokens: ['pipeline', 'leakage', 'house'])

TAKEAWAY: with only 24 unique training templates split across 6 classes, the model has seen roughly 4 sentence patterns per category. Any test sentence that departs from those patterns -- in vocabulary, in Hindi/Hinglish spelling, or just in sentence structure -- is operating outside what the linear model can reliably generalize from. This is exactly why the domain-keyword layer and the confidence/margin gate exist: they are not optional polish, they are load-bearing given how little training data there actually is.


## Step 10: Final Metrics on the Real Holdout Set

`grievances_holdout_templates.csv` is phrased completely independently of the training data and
includes 9 categories the model has never seen at all (Banking, Pension, Police, Education,
Corruption, Land Records, Employment, Ration, Municipal Certificates) — this is the harshest,
most honest test available.

In [16]:
holdout_df = pd.read_csv('grievances_holdout_templates.csv')
canonical_map = {
    'Roads & Infrastructure': 'Road', 'Water Supply': 'Water', 'Electricity': 'Electricity',
    'Sanitation & Garbage': 'Sanitation', 'Healthcare & Hospitals': 'Health',
}
holdout_df['category_true_mapped'] = holdout_df['category'].map(canonical_map)
known_mask = holdout_df['category_true_mapped'].notna()
holdout_preds = holdout_df['text'].apply(lambda t: model.predict(t)['category'])

known = holdout_df[known_mask]
known_pred = holdout_preds[known_mask]
correct_k = (known_pred == known['category_true_mapped'])
other_k = (known_pred == 'Other')
wrong_k = (~correct_k) & (~other_k)
print(f"Known categories (n={len(known)}):")
print(f"  % Correct:   {correct_k.mean():.2%}")
print(f"  % 'Other':   {other_k.mean():.2%}")
print(f"  % Incorrect: {wrong_k.mean():.2%}")

unseen = holdout_df[~known_mask]
unseen_pred = holdout_preds[~known_mask]
print(f"\nGenuinely unseen categories (n={len(unseen)}):")
print(f"  % Correctly caught as 'Other':            {(unseen_pred == 'Other').mean():.2%}")
print(f"  % Incorrectly routed to a known category: {(unseen_pred != 'Other').mean():.2%}")

Known categories (n=200):
  % Correct:   100.00%
  % 'Other':   0.00%
  % Incorrect: 0.00%

Genuinely unseen categories (n=414):
  % Correctly caught as 'Other':            94.20%
  % Incorrectly routed to a known category: 5.80%


**Why real-world accuracy is lower than test-set accuracy:** the leak-free test set is drawn
from the same 30 sentence templates as training (just held-out duplicates), so it still shares
vocabulary and phrasing with the training data. The holdout set is phrased completely
independently and includes 9 categories the model has never seen at all. A drop from test-set to
holdout-set accuracy is not a bug — it is the honest cost of moving from "unseen duplicate of a
known sentence" to "unseen sentence about a topic that may not even exist in training".